# Robustness, Perturbation, and Error Diagnostics Lab


## Purpose

This lab simulates robustness and grouped diagnostics for computer vision systems.

Aggregate accuracy can hide failures across lighting, camera, domain, or subgroup conditions.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1500

data = pd.DataFrame({
    "image_group": rng.choice(["A", "B", "C"], size=n, p=[0.5, 0.3, 0.2]),
    "lighting_condition": rng.choice(["normal", "low_light", "harsh_light"], size=n),
    "target": rng.binomial(1, 0.4, size=n),
})

lighting_error = data["lighting_condition"].map({
    "normal": 0.08,
    "low_light": 0.18,
    "harsh_light": 0.14,
})

group_error = data["image_group"].map({
    "A": 1.00,
    "B": 1.20,
    "C": 1.35,
})

error_probability = np.minimum(lighting_error * group_error, 0.90)
is_error = rng.binomial(1, error_probability)

data["prediction"] = np.where(is_error == 1, 1 - data["target"], data["target"])
data["error"] = data["prediction"] != data["target"]

summary = (
    data
    .groupby(["image_group", "lighting_condition"], as_index=False)
    .agg(
        sample_size=("error", "size"),
        classification_error_rate=("error", "mean"),
    )
)

summary

In [ ]:
summary.to_csv("../outputs/notebook_vision_error_diagnostics.csv", index=False)
summary

## Governance Extension

Before deployment, ask:

- Which conditions show the highest error?
- Are those conditions common in deployment?
- Are vulnerable groups or critical contexts affected?
- Is human review available?
- What monitoring will detect model degradation?